In [21]:
import pandas as pd

In [28]:
def preprocess_data(df):
    """
    Hàm tiền xử lý dữ liệu theo 4 bước yêu cầu.
    Đầu vào: df (Pandas DataFrame chứa dữ liệu gốc)
    """
    # Tạo một bản sao để không làm ảnh hưởng đến dữ liệu gốc
    df_clean = df.copy()

    # a. Remove null values (Xóa các dòng chứa giá trị rỗng/NA)
    df_clean = df_clean.dropna()

    # b. Remove duplicate instances (Xóa các dòng trùng lặp)
    # Khuyến nghị: Chỉ cần trùng lặp thời gian và mã cổ phiếu là tính là trùng
    df_clean = df_clean.drop_duplicates(subset=['datetime', 'symbol'])

    # Bước đệm: Ép kiểu cột datetime từ String (chuỗi) sang dạng Datetime của Pandas
    df_clean['datetime'] = pd.to_datetime(df_clean['datetime'])

    # c. Ensure the order of data (Sắp xếp dữ liệu theo thứ tự thời gian tăng dần)
    df_clean = df_clean.sort_values(by='datetime', ascending=True)

    # d. Convert string date-time to numerical timestamp 
    # (Chuyển đổi thời gian thành con số dạng Unix timestamp - tính bằng giây)
    df_clean['timestamp'] = df_clean['datetime'].apply(lambda x: int(x.timestamp()))

    # (Tùy chọn) Reset lại index của bảng cho gọn gàng sau khi xóa dòng và sắp xếp
    df_clean = df_clean.reset_index(drop=True)

    return df_clean

def calculate_ema5(df):
    """
    Hàm tính toán đường trung bình động hàm mũ EMA5.
    Đầu vào: df (DataFrame đã được tiền xử lý và sắp xếp theo thời gian)
    """
    df_ema = df.copy()
    
    # Tính EMA5 cho cột 'close'
    # span=5: Chu kỳ 5 nến
    # adjust=False: Sử dụng công thức đệ quy tiêu chuẩn (giống hệ thống TradingView)
    df_ema['EMA5'] = df_ema['close'].ewm(span=5, adjust=False).mean()
    
    return df_ema

In [23]:
df_raw = pd.read_csv('000001.csv')

In [24]:
df_raw

,datetime,symbol,open,high,low,close,volume
0,2025-02-05 08:30:00,SSE:000001,3270.2165,3271.0220,3235.1495,3239.8498,0.0
1,2025-02-05 08:45:00,SSE:000001,3239.8219,3245.6147,3234.7170,3235.3900,0.0
2,2025-02-05 09:00:00,SSE:000001,3235.1355,3237.9755,3228.1920,3237.5302,0.0
3,2025-02-05 09:15:00,SSE:000001,3237.4104,3241.3216,3237.2230,3239.5466,0.0
4,2025-02-05 09:30:00,SSE:000001,3239.6928,3240.0350,3234.4274,3235.5287,0.0
...,...,...,...,...,...,...,...
5107,2026-04-03 13:00:00,SSE:000001,3889.4670,3894.2031,3882.0730,3887.6599,0.0
5108,2026-04-03 13:15:00,SSE:000001,3887.3007,3888.0206,3881.0692,3881.5956,0.0
5109,2026-04-03 13:30:00,SSE:000001,3881.5191,3883.6392,3876.5250,3879.1429,0.0
5110,2026-04-03 13:45:00,SSE:000001,3879.7079,3881.7887,3879.1400,3881.5309,0.0


In [25]:
df_clean = preprocess_data(df_raw)
df_clean.head()

,datetime,symbol,open,high,low,close,volume,timestamp
0,2025-02-05 08:30:00,SSE:000001,3270.2165,3271.0220,3235.1495,3239.8498,0.0,1738744200
1,2025-02-05 08:45:00,SSE:000001,3239.8219,3245.6147,3234.7170,3235.3900,0.0,1738745100
2,2025-02-05 09:00:00,SSE:000001,3235.1355,3237.9755,3228.1920,3237.5302,0.0,1738746000
3,2025-02-05 09:15:00,SSE:000001,3237.4104,3241.3216,3237.2230,3239.5466,0.0,1738746900
4,2025-02-05 09:30:00,SSE:000001,3239.6928,3240.0350,3234.4274,3235.5287,0.0,1738747800


In [20]:
import pandas as pd

dt = pd.to_datetime("2025-02-05 08:30:00")
print(int(dt.timestamp()))

1738744200


In [29]:
df_ema = calculate_ema5(df_clean)
df_ema.head()

,datetime,symbol,open,high,low,close,volume,timestamp,EMA5
0,2025-02-05 08:30:00,SSE:000001,3270.2165,3271.0220,3235.1495,3239.8498,0.0,1738744200,3239.849800
1,2025-02-05 08:45:00,SSE:000001,3239.8219,3245.6147,3234.7170,3235.3900,0.0,1738745100,3238.363200
2,2025-02-05 09:00:00,SSE:000001,3235.1355,3237.9755,3228.1920,3237.5302,0.0,1738746000,3238.085533
3,2025-02-05 09:15:00,SSE:000001,3237.4104,3241.3216,3237.2230,3239.5466,0.0,1738746900,3238.572556
4,2025-02-05 09:30:00,SSE:000001,3239.6928,3240.0350,3234.4274,3235.5287,0.0,1738747800,3237.557937


In [32]:
features_cols = ['open', 'high', 'low', 'close', 'volume', 'EMA5']
df_ema[features_cols].values

array([[3270.2165    , 3271.022     , 3235.1495    , 3239.8498    ,
           0.        , 3239.8498    ],
       [3239.8219    , 3245.6147    , 3234.717     , 3235.39      ,
           0.        , 3238.3632    ],
       [3235.1355    , 3237.9755    , 3228.192     , 3237.5302    ,
           0.        , 3238.08553333],
       ...,
       [3881.5191    , 3883.6392    , 3876.525     , 3879.1429    ,
           0.        , 3883.03279991],
       [3879.7079    , 3881.7887    , 3879.14      , 3881.5309    ,
           0.        , 3882.53216661],
       [3880.0967    , 3880.0967    , 3880.0967    , 3880.0967    ,
           0.        , 3881.72034441]], shape=(5112, 6))

,datetime,symbol,open,high,low,close,volume,timestamp,EMA5
7,2025-02-05 10:15:00,SSE:000001,3237.8244,3241.9277,3237.2607,3239.4930,0.0,1738750500,3238.555544
8,2025-02-05 10:30:00,SSE:000001,3239.6090,3239.6596,3238.5496,3238.8520,0.0,1738751400,3238.654363
9,2025-02-05 12:00:00,SSE:000001,3239.4392,3239.4392,3228.2963,3233.4841,0.0,1738756800,3236.930942
10,2025-02-05 12:15:00,SSE:000001,3234.0345,3234.0345,3229.8374,3230.4175,0.0,1738757700,3234.759795
11,2025-02-05 12:30:00,SSE:000001,3230.9131,3231.6257,3223.7775,3223.7775,0.0,1738758600,3231.099030
...,...,...,...,...,...,...,...,...,...
5107,2026-04-03 13:00:00,SSE:000001,3889.4670,3894.2031,3882.0730,3887.6599,0.0,1775221200,3886.668825
5108,2026-04-03 13:15:00,SSE:000001,3887.3007,3888.0206,3881.0692,3881.5956,0.0,1775222100,3884.977750
5109,2026-04-03 13:30:00,SSE:000001,3881.5191,3883.6392,3876.5250,3879.1429,0.0,1775223000,3883.032800
5110,2026-04-03 13:45:00,SSE:000001,3879.7079,3881.7887,3879.1400,3881.5309,0.0,1775223900,3882.532167
